In [ ]:
!pip install ultralytics kagglehub -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 29.9 MB/s eta 0:00:0000:01


In [2]:
import kagglehub

path = kagglehub.dataset_download("rkuo2000/uecfood256")

print("Dataset path:", path)


Using Colab cache for faster access to the 'uecfood256' dataset.
Dataset path: /kaggle/input/uecfood256


In [4]:
import os
import shutil

src = path + "/UECFOOD256"
dst = "/content/food_dataset"

os.makedirs(dst, exist_ok=True)

# train + val klasörleri oluştur
for split in ["train", "val"]:
    os.makedirs(f"{dst}/{split}", exist_ok=True)

# klasörleri al
classes = sorted(os.listdir(src))

for split in ["train", "val"]:
    for i, cls in enumerate(classes):
        cls_path = os.path.join(src, cls)

        # bazı klasörler klasör olmayabilir -> atla
        if not os.path.isdir(cls_path):
            continue

        new_cls_path = f"{dst}/{split}/{i}"
        os.makedirs(new_cls_path, exist_ok=True)

        images = os.listdir(cls_path)

        # train / val split
        split_point = int(len(images) * 0.8)

        selected = images[:split_point] if split == "train" else images[split_point:]

        for img in selected:
            src_img = os.path.join(cls_path, img)
            dst_img = os.path.join(new_cls_path, img)

            # sadece dosyaları kopyala
            if os.path.isfile(src_img):
                shutil.copy(src_img, dst_img)


In [5]:
!find /content/food_dataset -type d | wc -l

515


In [6]:
import os

base = "/content/food_dataset"

for root, dirs, files in os.walk(base):
    for d in dirs:
        path = os.path.join(root, d)
        if len(os.listdir(path)) == 0:
            os.rmdir(path)

In [7]:
import os

train_classes = os.listdir("/content/food_dataset/train")
val_classes = os.listdir("/content/food_dataset/val")

print("Train:", len(train_classes))
print("Val:", len(val_classes))

Train: 256
Val: 256


In [10]:
from ultralytics import YOLO

model = YOLO("yolov8s-cls.pt")  # daha güçlü model

model.train(
    data="/content/food_dataset",
    epochs=50,
    imgsz=224
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/food_dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7832bb04e420>
curves: []
curves_results: []
fitness: 0.7971354424953461
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.6899174451828003, 'metrics/accuracy_top5': 0.9043534398078918, 'fitness': 0.7971354424953461}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.10475392005052675, 'inference': 0.5756778078965838, 'loss': 0.000369152452680904, 'postprocess': 0.0012487876668136194}
top1: 0.6899174451828003
top5: 0.9043534398078918

In [3]:
import os
from ultralytics import YOLO

# 1. Dosyayı otomatik bulalım
target_file = "test.jpg"
found_path = None

for root, dirs, files in os.walk('/'):
    if target_file in files:
        found_path = os.path.join(root, target_file)
        break

if found_path:
    print(f"✅ Dosya bulundu: {found_path}")
    
    # 2. Model Yükleme
    model_path = "/content/runs/classify/train/weights/best.pt"
    if os.path.exists(model_path):
        model = YOLO(model_path)
        
        # 3. Tahmin
        results = model(found_path)
        probs = results[0].probs
        names = model.names

        print("-" * 30)
        print(f"Tahmin: {names[probs.top1]}")
        print(f"Güven: {probs.top1conf:.4f}")
        print("-" * 30)
    else:
        print("❌ Model dosyası (.pt) belirtilen yolda yok!")
else:
    print("❌ 'test.jpg' sistemde hiçbir yerde bulunamadı. Lütfen dosyayı tekrar yükle.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
❌ 'test.jpg' sistemde hiçbir yerde bulunamadı. Lütfen dosyayı tekrar yükle.


In [5]:
# Eğer eski klasörü silmek istersen bu kodu bir kez çalıştır
!rm -rf /content/food_dataset

In [6]:
import os
import shutil

src = path + "/UECFOOD256"
dst = "/content/food_dataset"
# category.txt dosyasının yolu (Genellikle UECFOOD256 ana dizinindedir)
cat_file = path + "/UECFOOD256/category.txt"

# Kategori isimlerini oku (Eğer dosya varsa)
id_to_name = {}
if os.path.exists(cat_file):
    with open(cat_file, 'r') as f:
        lines = f.readlines()[1:] # İlk satır başlık olabilir, atla
        for line in lines:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                id_to_name[parts[0]] = parts[1].replace(" ", "_")

os.makedirs(dst, exist_ok=True)

for split in ["train", "val"]:
    os.makedirs(f"{dst}/{split}", exist_ok=True)

classes = sorted(os.listdir(src))

for cls in classes:
    cls_path = os.path.join(src, cls)
    if not os.path.isdir(cls_path) or cls == "labels":
        continue

    # İSİM BURADA BELİRLENİYOR:
    # Eğer listede varsa ismini koy, yoksa klasörün kendi adını (örn: '89') koy
    folder_name = id_to_name.get(cls, cls) 

    images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    split_point = int(len(images) * 0.8)

    for split in ["train", "val"]:
        new_cls_path = os.path.join(dst, split, folder_name)
        os.makedirs(new_cls_path, exist_ok=True)
        
        selected = images[:split_point] if split == "train" else images[split_point:]
        
        for img in selected:
            shutil.copy(os.path.join(cls_path, img), os.path.join(new_cls_path, img))

print("Veri seti isimlerle hazırlandı!")

Veri seti isimlerle hazırlandı!


In [ ]:
from ultralytics import YOLO

# 1. Medium modeli çekiyoruz (Sıfırdan veya pretrained olarak)
model = YOLO("yolov8m-cls.pt") 

# 2. Eğitimi başlat
model.train(
    data="/content/food_dataset",
    epochs=100,
    imgsz=224,
    batch=32,
    patience=20,
    save=True,
    device=0,
    optimizer='AdamW',
    lr0=0.001
)

Ultralytics 8.4.35 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/food_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=20, perspective=0.0,

: 